In [19]:
data["MCQA"]

{'question': "Which of the following statements about A's stock price and the given financial analysis is incorrect?\nA. Based solely on the immediate post-news trading session, the stock's price trajectory predicts a return to its previous high of 138.92 within the next week, as investor confidence remains unshaken.\nB. The stock price fluctuations after the news publication indicate heightened volatility, with the price descending from close to 131 shortly after the news release, which could suggest negative market sentiment in response to the news.\nC. The stock price decreased significantly from the last observed price of 138.22 on 2022-09-16 to a low of 128.48 shortly after the published news, indicating a downward trend following the announcement.\nD. The news content focuses on the potential of Agilent Technologies as part of the Zacks Focus List, which historically has shown to outperform the market; therefore, it is plausible that the stock could eventually rebound after an in

In [18]:
data.keys()

dict_keys(['input_timestamps', 'input_window', 'output_timestamps', 'output_window', 'text', 'timestamp_ms', 'published_utc', 'article_url', 'news_price_correlation', 'MCQA', 'timeseries'])

In [ ]:
syn_question = f"""
You need to answer this question based on the time series: {data["MCQA"]["question"]},Choose the most appropriate option and return ABCD.  \n
This is news that was published right after the time series data ended. You need to refer to this news information to make your selection.:{news}. \n
"""

In [ ]:
from adjustText import adjust_text
plt.plot(data["timeseries"])
plt.gcf().autofmt_xdate()
plt.savefig("output.png")

In [ ]:
additional_args={"timeseries":data["timeseries"]}
tidy_args = additional_args.copy()
tidy_args_type = analyze_dict_types(tidy_args)
tidy_args = tidy_args_to_str(tidy_args)
tidy_args_type
        

llm_checker = LLM_checker("The response should include explanations of zero or more temporal nouns, and should not contain answers to other questions or code.")
promt=f"""
    Your task is to check whether it involves multiple temporal terms that need to be distinguished such as 'downward spike', especially in the options. If so, use the search_tool to search for all of them. The usage method is, for example, "What are the differences between these temporal terms xx, xx, xx?"
    The following will present a question and options. 
    the question is {data["question"]},
    Note that you don't need to answer this question, just analyze the nouns within it.
    The workflow is as follows: first, check if there are any temporal terms that require distinction in meaning, then use the search_tool to ask all questions at once, and finally organize the search results. The final returned result should include explanations of multiple terms.
    """
pre_analysis_agent.final_answer_checks = [llm_checker.check]
Term_search = pre_analysis_agent.run(promt)
pre_analysis_agent.final_answer_checks = []

In [ ]:
promt=f"""
the question is {syn_question}. The time series data has been represented as images.
the data is {tidy_args},
Select the most appropriate option based on the image and news, and refer to the preceding task description for this time sequence.
"""
fig_anaysis_tool = FigAnaysisTool()
res = fig_anaysis_tool(promt,"output.png")
fig_info = res[0].message.content

In [ ]:
fig_info

In [ ]:
promt=f"""
    The following will present a question and options. You need to extract the time-series operations involved in the question and options. For example, if the option requires analyzing noise, then search for how to extract temporal noise and how to determine whether the noise follows a specific distribution? Then, use the search_tool to find methods for performing these operations. Finally, summarize the results into a dictionary "operation name": "string(content and precautions)".
    question and options:

    the question is {syn_question},
    the data is {tidy_args},
    You don't need to answer questions, just return the sample code corresponding to the task
    The answer obtained from the image is {fig_info},
    The search results may contain multiple methods and extensive content. Please select the most suitable method for each task and moderately streamline the content. The response must include the task and a code example.
    The workflow is as follows: first, determine the sequence of operations required based on the task, then query the search_tool, and finally simplify and summarize the final answer. The final answer must include the task and example code.  
    To speed up the search process, try to search for multiple items at once whenever possible, up to a maximum of three.
    """

pre_res = pre_analysis_agent.run(promt)


In [ ]:
prompt= f"""This is the format requirement for the  answers:{syn_question}
 You need to check whether the answers provided later strictly comply with the format requirements for the answers in the questions, including line breaks, etc.
 """
llm_checker = LLM_checker(prompt)
prompt=f"""
        the question is {syn_question},\n
        Here are some methods for reference{pre_res}.\n
        The answer obtained from the image is {fig_info}.\n
        Your task is to answer questions based on image analysis results and code. You need to first determine whether to directly adopt the image's conclusions or use the image to assist in answering with code. For problems involving only one or two sequences that can be resolved visually, such as local characteristic fluctuations, you can directly adopt the image's conclusions. For issues involving multiple sequences, code analysis is typically required. For questions where the image response is "not clear," you can only reply with code.\n
        You have been provided with these additional arguments, that you can access using the keys as variables in your python code:{tidy_args},the type of the values is {tidy_args_type}。The "timeseries" data is a DataFrame where each column represents a data series.
        These variables have been loaded into the interpreter's memory. You can only call them and must not overwrite these variables.And you cannot draw or search the web.
        Note that when asked about the existence of a phenomenon, you need to compare its magnitude with the overall mean to determine if it is significant. If it is not significant (e.g., <5%), it can be considered non-existent.
        Answer questions cautiously based on the provided news.
        """

data_analysis_agent.state.update(additional_args)
data_analysis_agent.final_answer_checks = [llm_checker.check]
da_res = (data_analysis_agent.run(prompt))

In [ ]:
print(data["MCQA"]["answer"])

In [ ]:
data.keys()